In [3]:
import math
from typing import List, Tuple, Dict
from collections import defaultdict


def calculate_perplexity(sentences: List[List[str]], ngram_counts: Dict,
                        prev_ngram_counts: Dict, vocabulary: set,
                        n: int, k: float = 1.0) -> float:
    """
    Calculate perplexity score for sentences.

    Perplexity = exp(-1/N * sum(log(P(w_i | prev_ngram))))
    """
    vocab_size = len(vocabulary)
    total_log_prob = 0.0
    total_words = 0

    for sent in sentences:
        padded = ['<s>'] * (n - 1) + sent + ['<e>']

        for i in range(n - 1, len(padded)):
            prev_ngram = tuple(padded[i-n+1:i])
            word = padded[i]

            full_ngram = prev_ngram + (word,)
            numerator = ngram_counts.get(full_ngram, 0) + k
            denominator = prev_ngram_counts.get(prev_ngram, 0) + k * vocab_size

            prob = numerator / denominator if denominator > 0 else k / (k * vocab_size)
            total_log_prob += math.log(prob)
            total_words += 1

    if total_words == 0:
        return float('inf')

    return math.exp(-total_log_prob / total_words)


def evaluate_model(test_sentences: List[List[str]], ngram_counts: Dict,
                  prev_ngram_counts: Dict, vocabulary: set, n: int) -> float:
    """Evaluate model using perplexity."""
    return calculate_perplexity(test_sentences, ngram_counts, prev_ngram_counts,
                               vocabulary, n)